# 🚀 Модуль 9. Урок 3. Фреймворк валидации AI-ответов и защита от галлюцинаций

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_09_ai_tools_daily/lesson_03_ai_validation_framework_hallucinations.ipynb)

## 📌 Бизнес-контекст и методология
Аналитик несет юридическую и финансовую ответственность за презентуемые цифры.
Фреймворк фактчекинга:
1. Запрет на слепое копирование чисел из текстового ответа LLM.
2. Проверка граничных условий и сумм (Sanity Checks).
3. Дублирующий расчет на агрегированном SQL.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка ключевых датасетов
customers_csv = ensure_dataset("customers.csv", subfolder="raw")
orders_csv = ensure_dataset("orders.csv", subfolder="raw")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
print("🟢 Окружение готово к анализу данных.")


🟢 Окружение готово к анализу данных.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
import pandas as pd

# Пайплайн автоматической верификации баланса
def verify_financial_integrity(df_transactions, reported_total):
    calculated_total = df_transactions['amount'].sum()
    discrepancy = abs(calculated_total - reported_total)
    is_valid = discrepancy < 0.01
    return is_valid, discrepancy

sample_tx = pd.DataFrame({'amount': [100.50, 200.25, 45.25]})
valid, diff = verify_financial_integrity(sample_tx, 346.00)

print(f"Баланс сошелся: {valid} (Расхождение: ${diff:.4f})")


Баланс сошелся: True (Расхождение: $0.0000)


### 🎯 Практическое задание 1: Детекция расхождения в балансе
Проверьте реакцию валидатора на сфальсифицированную сумму 400.00.

In [3]:
### ВАШ КОД ЗДЕСЬ
bad_valid, bad_diff = verify_financial_integrity(sample_tx, 400.00)
assert not bad_valid, "Валидатор должен зафиксировать ошибку"
assert bad_diff > 50.0, "Расхождение должно быть обнаружено"
print("✅ Тест успешно пройден: валидатор целостности надежен!")


✅ Тест успешно пройден: валидатор целостности надежен!


### 💡 Золотое правило Data Analyst 2027
AI генерирует гипотезы и черновики кода, но валидируют результат автотесты и математические инварианты.